# Expériences : du 0,809 au 0,793 (métadonnées des films)

**DataLab A1, groupe 16.** Ce notebook reproduit les expériences qui ont mené à la soumission à **RMSE 0,793** sur la plateforme.

| Soumission | RMSE plateforme | Temps |
|---|---|---|
| Voisinage (cours) | 0,951 | 1,7 s |
| Factorisation (cours) | 0,872 | 2,6 s |
| Bayesian SVD++ flipped, 8 chaînes (A) | 0,809 | 119 s |
| + a priori myFM réglés à l'échelle de la plateforme (B) | 0,806 | 137 s |
| **+ genres et période de sortie (un seul modèle, rang 42)** | **0,793** | 299 s |

**Plan**
1. Données et protocole « plateforme ».
2. Le modèle : Bayesian SVD++ flipped (myFM) + informations sur les films.
3. Expérience principale : effet de chaque ajout, à protocole identique.
4. Nombre de chaînes.
5. Recherche d'hyperparamètres (résumé des scripts Optuna).
6. Test final en local (train → test), une seule fois.

Fichiers attendus dans `DATA` : `ratings_train.npy`, `ratings_test.npy`, `namesngenre2.npy`. Durée totale : environ 12 min sur 2 cœurs.

In [1]:
import os, re, time, warnings
os.environ.setdefault("TQDM_DISABLE", "1")
warnings.filterwarnings("ignore")
import numpy as np, scipy.sparse as sp, pandas as pd
import myfm

DATA = os.environ.get("A1_DATA", "..")                     # dossier des données (le dépôt, depuis notebooks/)
TR = np.load(f"{DATA}/ratings_train.npy").T                  # films x utilisateurs, NaN = inconnu
TE = np.load(f"{DATA}/ratings_test.npy").T
META = np.load(f"{DATA}/namesngenre2.npy")                   # ligne j = colonne j des notes : [titre, genres]
m, n = TR.shape
print(f"{m} films x {n} utilisateurs | train {np.sum(~np.isnan(TR))} notes | test {np.sum(~np.isnan(TE))} notes")
print("notes par film (médiane) :", int(np.median((~np.isnan(TR)).sum(1))), "| par utilisateur :", int(np.median((~np.isnan(TR)).sum(0))))

4980 films x 610 utilisateurs | train 31598 notes | test 31598 notes
notes par film (médiane) : 3 | par utilisateur : 24


## 1. Protocole « plateforme »

La plateforme entraîne sur plus de données que notre train (même modèle : 0,850 en local, 0,809 sur la plateforme). Pour régler les hyperparamètres à la bonne échelle, on imite cette situation :

- on coupe le test en deux moitiés aléatoires H1 et H2 (graine 0) ;
- on entraîne sur **train + H1** (1,5 fois les données) et on mesure sur **H2**, puis l'inverse ;
- le score est la moyenne des deux RMSE.

Chaque réglage est évalué avec **4 chaînes MCMC moyennées** (graines 2 à 5). Ces graines n'ont jamais servi pendant les recherches Optuna, qui utilisaient les graines 0 et 1. Avec une seule chaîne, le bruit de graine (±0,0015) est du même ordre que les écarts entre réglages.

Ce protocole utilise le test : le test local n'est donc plus « propre » pour ces réglages. On garde une mesure honnête à la section 6, train → test, sans rien régler dessus.

In [2]:
def halves(seed=0):
    ii, uu = np.where(~np.isnan(TE))
    perm = np.random.default_rng(seed).permutation(len(ii)); h = len(ii) // 2
    out = []
    for a, b in [(perm[:h], perm[h:]), (perm[h:], perm[:h])]:
        R = TR.copy(); R[ii[a], uu[a]] = TE[ii[a], uu[a]]          # train + une moitié du test
        out.append((R, ii[b], uu[b], TE[ii[b], uu[b]]))           # mesure sur l'autre moitié
    return out

rmse = lambda p, y: float(np.sqrt(np.mean((np.clip(p, 0.5, 5) - y) ** 2)))
H = halves()
print([len(h[3]) for h in H], "notes mesurées par moitié")

[15799, 15799] notes mesurées par moitié


## 2. Le modèle

**Bayesian SVD++ flipped** (Rendle, Zhang & Koren 2019) : une *factorization machine* apprise par échantillonnage de Gibbs avec myFM. Chaque note est décrite par deux blocs de colonnes :

- **utilisateur** = [identifiant | films qu'il a notés, normalisés par $|N(u)|^{-e_u}$] ;
- **film** = [identifiant | utilisateurs qui l'ont noté, normalisés par $|N(i)|^{-e_i}$ | *genres* | *période de sortie* (options)].

La prédiction est la somme des termes linéaires et de toutes les interactions deux à deux entre ces colonnes, moyennée sur les tirages MCMC. Il n'y a pas de pénalité à régler : la régularisation est elle-même tirée a posteriori, à partir des a priori `alpha_0, beta_0, gamma_0, reg_0`.

**Informations sur les films** (`namesngenre2.npy`, MovieLens) :
- genres : 20 colonnes 0/1, normalisées pour que chaque film ait une ligne de norme 1 ;
- année : lue dans le titre (« Toy Story (1995) »), puis rangée par tranches. Un seul film n'a pas d'année (*Ready Player One*) : on lui donne la médiane.

In [3]:
GENRES = sorted({g for s in META[:, 1] for g in s.split("|")})
G = np.array([[g in s.split("|") for g in GENRES] for s in META[:, 1]], float)
G /= np.sqrt(np.maximum(G.sum(1, keepdims=True), 1))
year = np.array([float(mt.group(1)) if (mt := re.search(r"\((\d{4})\)\s*$", t.strip())) else np.nan for t in META[:, 0]])
year = np.where(np.isnan(year), np.nanmedian(year), year)
onehot = lambda b, k: np.eye(k)[b]
PERIODS = {"10": onehot(np.digitize(year, [1960, 1970, 1980, 1990, 1995, 2000, 2005, 2010, 2015]), 10),   # surtout par décennies
           "5": onehot(np.digitize(year, list(range(1950, 2020, 5))), 15)}                                  # tranches de 5 ans
print(len(GENRES), "genres | années", int(year.min()), "-", int(year.max()))


def chain(R, ti, tu, rank, n_iter, seed, priors, ex_u=0.5, ex_i=0.5, genres=False, period=None):
    # Une chaîne MCMC ; renvoie les prédictions aux cases (ti, tu).
    M = ~np.isnan(R); ii, uu = np.where(M)
    norm = lambda A, e: sp.diags(np.maximum(np.asarray(A.sum(1)).ravel(), 1) ** -e) @ A
    Mu, Mi = sp.csr_matrix(M.T.astype(float)), sp.csr_matrix(M.astype(float))
    U, gu = [sp.identity(n, format="csr"), norm(Mu, ex_u)], [n, m]                 # bloc utilisateur
    I, gi = [sp.identity(m, format="csr"), norm(Mi, ex_i)], [m, n]                 # bloc film
    if genres:  I.append(sp.csr_matrix(G)); gi.append(G.shape[1])
    if period:  I.append(sp.csr_matrix(PERIODS[period])); gi.append(PERIODS[period].shape[1])
    Xu, Xi = sp.hstack(U).tocsr(), sp.hstack(I).tocsr()
    fm = myfm.MyFMRegressor(rank=rank, init_stdev=0.1, random_seed=seed, **priors)
    fm.fit(sp.csr_matrix((len(ii), 0)), R[ii, uu], X_rel=[myfm.RelationBlock(uu, Xu), myfm.RelationBlock(ii, Xi)],
           n_iter=n_iter, group_shapes=gu + gi)
    # prédiction calculée à partir des paramètres de chaque tirage (bien plus rapide que fm.predict sur des blocs)
    pu = Xu.shape[1]; Xu2, Xi2 = Xu.multiply(Xu).tocsr(), Xi.multiply(Xi).tocsr(); S = np.zeros(len(ti))
    for w0, w, V in zip(fm.w0_samples, fm.w_samples, fm.V_samples):
        Au, Bi = Xu @ V[:pu], Xi @ V[pu:]
        hu = 0.5 * ((Au ** 2).sum(1) - (Xu2 @ (V[:pu] ** 2)).sum(1))              # interactions dans le bloc utilisateur
        hi = 0.5 * ((Bi ** 2).sum(1) - (Xi2 @ (V[pu:] ** 2)).sum(1))              # interactions dans le bloc film
        S += w0 + (Xu @ w[:pu] + hu)[tu] + (Xi @ w[pu:] + hi)[ti] + (Bi[ti] * Au[tu]).sum(1)
    return S / len(fm.w0_samples)


def evaluate(cfg, seeds=range(2, 6)):
    # Protocole plateforme : moyenne des chaînes, RMSE sur chaque moitié. Renvoie aussi les prédictions.
    preds = [[chain(R, ti, tu, seed=s, **cfg) for s in seeds] for R, ti, tu, y in H]
    return [rmse(np.mean(p, 0), h[3]) for p, h in zip(preds, H)], preds

20 genres | années 1902 - 2018


## 3. Expérience principale : effet de chaque ajout

Même protocole, mêmes graines, un seul changement à chaque ligne.

| Ligne | Ce qui change |
|---|---|
| A | Bayesian SVD++ flipped, rang 16, a priori par défaut de myFM (tous à 1), exposants 1/2 |
| B | A priori et exposants réglés sans métadonnées (Optuna) |
| C | + genres |
| D | + période de sortie (tranches surtout décennales) |
| D′ | D, avec des tranches de 5 ans |
| D″ | D′, avec les a priori réglés de nouveau en présence des métadonnées (rang 16) |
| E | D″ avec le rang 42 : **la soumission à 0,793** |

In [4]:
DEFAULT = dict(alpha_0=1.0, beta_0=1.0, gamma_0=1.0, reg_0=1.0)
TUNED   = dict(alpha_0=0.885, beta_0=0.15, gamma_0=0.366, reg_0=7.806)   # réglés sans métadonnées
META_P  = dict(alpha_0=0.25, beta_0=0.016, gamma_0=0.416, reg_0=0.461)   # réglés avec métadonnées
CONFIGS = {
    "A  a priori par défaut":                  dict(rank=16, n_iter=150, priors=DEFAULT),
    "B  a priori réglés":                      dict(rank=16, n_iter=150, priors=TUNED, ex_u=0.373, ex_i=0.417),
    "C  + genres":                             dict(rank=16, n_iter=150, priors=TUNED, ex_u=0.373, ex_i=0.417, genres=True),
    "D  + période":                            dict(rank=16, n_iter=150, priors=TUNED, ex_u=0.373, ex_i=0.417, genres=True, period="10"),
    "D′ tranches de 5 ans":                    dict(rank=16, n_iter=150, priors=TUNED, ex_u=0.373, ex_i=0.417, genres=True, period="5"),
    "D″ a priori réglés de nouveau, rang 16":   dict(rank=16, n_iter=150, priors=META_P, ex_u=0.507, ex_i=0.498, genres=True, period="5"),
    "E  tranches 5 ans, rang 42 (0,793)":      dict(rank=42, n_iter=150, priors=META_P, ex_u=0.507, ex_i=0.498, genres=True, period="5"),
}
rows, PRED = [], {}
for name, cfg in CONFIGS.items():
    t = time.time(); r, PRED[name] = evaluate(cfg)
    rows.append(dict(config=name, H1=r[0], H2=r[1], RMSE=np.mean(r), temps_s=round(time.time() - t)))
    print(f"{name:<40} {np.mean(r):.4f}  ({time.time() - t:.0f} s)", flush=True)
res = pd.DataFrame(rows).set_index("config")
res["gain vs ligne précédente"] = res["RMSE"].diff()
res.round(4)

A  a priori par défaut                   0.8308  (99 s)


B  a priori réglés                       0.8274  (100 s)


C  + genres                              0.8158  (105 s)


D  + période                             0.8092  (107 s)


D′ tranches de 5 ans                     0.8095  (108 s)


D″ a priori réglés de nouveau, rang 16   0.8084  (107 s)


E  tranches 5 ans, rang 42 (0,793)       0.8077  (264 s)


,H1,H2,RMSE,temps_s,gain vs ligne précédente
config,,,,,
A a priori par défaut,0.8371,0.8245,0.8308,99,NaN
B a priori réglés,0.8333,0.8214,0.8274,100,-0.0034
C + genres,0.8198,0.8119,0.8158,105,-0.0116
D + période,0.8131,0.8053,0.8092,107,-0.0067
D′ tranches de 5 ans,0.8131,0.8058,0.8095,108,0.0003
"D″ a priori réglés de nouveau, rang 16",0.8118,0.8049,0.8084,107,-0.0011
"E tranches 5 ans, rang 42 (0,793)",0.8105,0.8048,0.8077,264,-0.0007


**Lecture**
- **A → B** : les a priori par défaut de myFM sont mal adaptés à des données aussi creuses. Les régler fait gagner environ 0,003 (sur la plateforme : 0,809 → 0,806).
- **B → C → D** : les genres puis la période de sortie font le plus gros gain du projet. La majorité des films n'ont que 1 à 3 notes : pour eux, savoir que c'est un drame des années 1990 compte presque autant que les notes elles-mêmes.
- **D → D′ → D″ → E** : un changement à la fois. Les tranches de 5 ans n'apportent rien en elles-mêmes ; régler de nouveau les a priori avec les métadonnées fait gagner un peu ; le rang 42 encore un peu, une fois ces a priori en place. E est la version soumise (0,793 sur la plateforme).

## 4. Nombre de chaînes

Moyenner des chaînes indépendantes réduit le bruit MCMC. Courbe pour la configuration E, sur les 4 chaînes déjà calculées. Dans nos scripts, aller au-delà de 8 chaînes (jusqu'à 16) ne gagnait plus que 0,0002 pour un temps doublé : on a gardé 8 chaînes.

In [5]:
P = PRED["E  tranches 5 ans, rang 42 (0,793)"]
curve = {k: np.mean([rmse(np.mean(P[h][:k], 0), H[h][3]) for h in range(2)]) for k in range(1, 5)}
pd.Series(curve, name="RMSE").rename_axis("chaînes").round(4)

chaînes
1    0.8101
2    0.8086
3    0.8082
4    0.8077
Name: RMSE, dtype: float64

## 5. Recherche d'hyperparamètres (résumé)

Les recherches ont été faites avec Optuna (TPE) sur le même protocole. Chaque essai est la moyenne de 2 chaînes (graines 0 et 1), et les meilleurs réglages ont ensuite été validés sur les graines 2 à 5, comme ci-dessus. Elles ne sont pas ré-exécutées ici, car elles prennent plusieurs heures. Le code est dans la cellule suivante : mettre `RUN_SEARCH = True` pour la relancer.

| Recherche | Espace exploré | Conclusion |
|---|---|---|
| Rang, itérations, burn-in, init (sans métadonnées) | rang 4–48, 100–500 it., burn-in 0–100 | rang 16, environ 150 it., pas de burn-in : déjà optimal. Le burn-in dégrade. |
| A priori myFM + exposants (sans métadonnées) | `alpha_0, beta_0, gamma_0, reg_0` ∈ [0,01 ; 100], exposants ∈ [0 ; 1,5] | `beta_0`, `gamma_0` petits, `reg_0` grand, exposants ≈ 0,4 : −0,003 |
| Tout, avec métadonnées | idem + rang 8–48 + tranches d'années | rang 42 : −0,0015 contre le rang 16 ; paysage plat sur les a priori |
| Tout, avec métadonnées (2e passe) | rang 24–80, 120–240 it. | rang 42 et 150 it. confirmés (250 it. : −0,0002 pour 1,7 fois le temps) |


In [6]:
RUN_SEARCH = False
if RUN_SEARCH:
    import optuna
    def objective(trial):
        cfg = dict(rank=trial.suggest_int("rank", 24, 80, log=True), n_iter=trial.suggest_int("n_iter", 120, 240, step=30),
                   priors={k: trial.suggest_float(k, 0.005, 100, log=True) for k in ("alpha_0", "beta_0", "gamma_0", "reg_0")},
                   ex_u=trial.suggest_float("ex_u", 0.2, 0.8), ex_i=trial.suggest_float("ex_i", 0.2, 0.8),
                   genres=True, period="5")
        return float(np.mean(evaluate(cfg, seeds=(0, 1))[0]))
    study = optuna.create_study(sampler=optuna.samplers.TPESampler(seed=0))
    study.optimize(objective, n_trials=40)
    print(study.best_params, study.best_value)

## 6. Test final en local : train → test, une seule fois

C'est exactement le modèle soumis à 0,793 (configuration E, 8 chaînes, graines 0 à 7), entraîné sur le train seul et mesuré une fois sur le test. Aucun réglage n'a été choisi sur cette mesure.

Pour comparer, la même mesure donnait 0,8501 pour le modèle à 0,809 (A).

In [7]:
ti, tu = np.where(~np.isnan(TE)); y = TE[ti, tu]
t = time.time()
cfgE = CONFIGS["E  tranches 5 ans, rang 42 (0,793)"]
pE = np.mean([chain(TR, ti, tu, seed=s, **cfgE) for s in range(8)], 0)
print(f"Configuration E, 8 chaînes : RMSE test = {rmse(pE, y):.4f}  ({time.time() - t:.0f} s)")

Configuration E, 8 chaînes : RMSE test = 0.8242  (213 s)


## Conclusion

- Le gain principal ne vient pas d'un modèle plus complexe, mais d'une **information en plus sur les films** (genres, époque) : c'est là que les données manquent le plus.
- Le modèle final reste **un seul modèle** : 8 chaînes, environ 300 s sur la plateforme (limite : 1 200 s).
- Les gains mesurés sur le protocole « plateforme » se sont retrouvés sur la plateforme (0,809 → 0,806 → 0,793), ce qui valide ce protocole pour régler les hyperparamètres.

*Source des métadonnées : MovieLens Latest Small (GroupLens). F. M. Harper & J. A. Konstan, « The MovieLens Datasets: History and Context », ACM TiiS 5(4), 2015.*